In [1]:
# !pip install flask pandas python-dotenv openpyxl supabase
# !pip install supabase
# !pip install chardet

In [2]:
# pip show supabase

In [3]:
import pandas as pd
import glob
import os
import io

In [5]:
#main file ko import kia:
import chardet

with open(r"D:\i\uploads\raw\ActivationDetailReport_7GT0TN4BG.csv", "rb") as f:
    result = chardet.detect(f.read())

print(result)

{'encoding': 'iso8859-3', 'confidence': 0.013813312491996025, 'language': 'eo', 'mime_type': 'text/plain'}


In [6]:
#main file ko ActivationDetailReport variable bna dia:
ActivationDetailReport = pd.read_csv(r"D:\i\uploads\raw\ActivationDetailReport_7GT0TN4BG.csv", encoding='latin1')


In [29]:
ActivationDetailReport

,marketid,storeid,company,custid,invno,serial,item,mobile,actdate,account,...,term,prodline,itmdesc,username,acttype,plancode,plantype1,grouptype,plandesc,mrc
0,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,82573,163115,'356553100153822,',5708027036,01/01/2026 13:15:00,117355760,...,NaN,NaN,NaN,BWH13560,Upgrade,50UNLG1,Voice,Individual,$50 UNL LTE W/ GOOGLE ONE,50.0
1,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,82573,163115,'356553100153822,',5708027036,01/01/2026 13:15:00,117355760,...,NaN,NaN,NaN,BWH13560,Upgrade,MBPY,Feature,Individual,Metro Basic Protection,6.0
2,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,82585,163127,'354445470898494,'610214685872,6465715972,01/02/2026 10:24:00,318493628,...,NaN,METROPCS,SAM A166U A16 5G 128G BLU KIT,CGN92584,New Activation,BFLX50,Voice,Individual,METRO FLEX UNLIMITED $50,55.0
3,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,82585,163127,'354445470898494,'610214685872,6465715972,01/02/2026 10:24:00,318493628,...,NaN,METROPCS,SAM A166U A16 5G 128G BLU KIT,CGN92584,New Activation,PSB2NY,Feature,Individual,Premium Service Bundle,10.0
4,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,82585,163127,'354445470898494,'610214685872,6465715972,01/02/2026 10:24:00,318493628,...,NaN,METROPCS,SAM A166U A16 5G 128G BLU KIT,CGN92584,New Activation,MEXUNL,Feature,Individual,MEXICO AND CANADA UNLIMITED,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,83680,164276,'352328628493420,'194252838174,3474075014,01/31/2026 17:41:00,482800720,...,NaN,METROPCS,APL IPHONE 13 128G BLK TMUS KIT,BWH13560,New Activation,BFLX60,Voice,Individual,METRO FLEX UNLIMITED PLUS $60,65.0
318,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,83680,164276,'352328628515156,'194252838174,3474086029,01/31/2026 17:41:00,482800720,...,NaN,METROPCS,APL IPHONE 13 128G BLK TMUS KIT,BWH13560,New Activation,BFLX60AAL,Voice,Individual,FLEX UNLIMITED PLUS $35 AAL,35.0
319,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,83680,164276,'353337690316418,'610214688811,3473763605,01/31/2026 17:42:00,482800720,...,NaN,METROPCS,SAM A176U A17 128G BLK KIT,BWH13560,New Activation,BFLX60L4,Voice,Individual,FLEX UNLIMITED PLUS $10 AAL,10.0
320,METRO-NY,C-70812973,DKS WIRELESS 2128007-DCWP,83680,164276,'353337690409171,'610214688811,3476380579,01/31/2026 17:42:00,482800720,...,NaN,METROPCS,SAM A176U A17 128G BLK KIT,BWH13560,New Activation,BFLX60AAL,Voice,Individual,FLEX UNLIMITED PLUS $35 AAL,35.0


In [7]:
#useful column ko alag variable me store kr dia:

useful_cols_ActivationDetailReport = ActivationDetailReport[['storeid','company','serial','accounttype','plancode','plantype1',]]

In [8]:
useful_cols_ActivationDetailReport

,storeid,company,serial,accounttype,plancode,plantype1
0,C-70812973,DKS WIRELESS 2128007-DCWP,'356553100153822,Prime,50UNLG1,Voice
1,C-70812973,DKS WIRELESS 2128007-DCWP,'356553100153822,Prime,MBPY,Feature
2,C-70812973,DKS WIRELESS 2128007-DCWP,'354445470898494,Prime,BFLX50,Voice
3,C-70812973,DKS WIRELESS 2128007-DCWP,'354445470898494,Prime,PSB2NY,Feature
4,C-70812973,DKS WIRELESS 2128007-DCWP,'354445470898494,Prime,MEXUNL,Feature
...,...,...,...,...,...,...
317,C-70812973,DKS WIRELESS 2128007-DCWP,'352328628493420,Prime,BFLX60,Voice
318,C-70812973,DKS WIRELESS 2128007-DCWP,'352328628515156,Prime,BFLX60AAL,Voice
319,C-70812973,DKS WIRELESS 2128007-DCWP,'353337690316418,Prime,BFLX60L4,Voice
320,C-70812973,DKS WIRELESS 2128007-DCWP,'353337690409171,Prime,BFLX60AAL,Voice


In [10]:
# sirf voice wale rows ko rakh lia baki ko hata dia:
useful_cols_ActivationDetailReport = useful_cols_ActivationDetailReport[useful_cols_ActivationDetailReport["plantype1"] == "Voice"]

In [24]:
# serial number me se non digit character ko hata dia:
useful_cols_ActivationDetailReport['serial'] = (useful_cols_ActivationDetailReport  ['serial'].astype(str).str.replace(r'\D', '', regex=True))

In [25]:
# serial number ko dtype=int me convert kr dia:
useful_cols_ActivationDetailReport['serial'] = (
    useful_cols_ActivationDetailReport['serial']
    .astype(int)
)

In [26]:
useful_cols_ActivationDetailReport.info()

<class 'pandas.DataFrame'>
Index: 218 entries, 0 to 321
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   storeid      218 non-null    str  
 1   company      218 non-null    str  
 2   serial       218 non-null    int64
 3   accounttype  218 non-null    str  
 4   plancode     218 non-null    str  
 5   plantype1    218 non-null    str  
dtypes: int64(1), str(5)
memory usage: 11.9 KB


In [27]:
# serial number ke count ko nikal lia:
serial_counts = useful_cols_ActivationDetailReport['serial'].value_counts().reset_index()

serial_counts.columns = ['serial', 'count']

print(serial_counts)

              serial  count
0    356115235577554      2
1    356553100153822      1
2    354445470898494      1
3    357917872023779      1
4     16758008762556      1
..               ...    ...
212  352328628493420      1
213  352328628515156      1
214  353337690316418      1
215  353337690409171      1
216  863289057359114      1

[217 rows x 2 columns]


In [28]:
# serial number ke count me se 1 se zyada wale ko nikal lia jisse pta chale kitne duplicate serial number hai:
duplicates = serial_counts[serial_counts['count'] > 1]

print(duplicates)

            serial  count
0  356115235577554      2


this shows that there is no duplicate serial number present in ActivationDetailReport
And Bhavna said thet it will not have any.

Now moving towards curCallidus report we need esn=serial for maping

In [29]:
# CurCallidusReport file ko CurCallidusReport variable me import kia:
CurCallidusReport = pd.read_csv(r"D:\i\uploads\raw\curCallidus_7GT0TR7AH.csv", encoding='latin1')

In [57]:
# ye chargeback ke liye transaction_type_desc me se charge word ko search kr ke ek dictionary bna rha hai jisme esn ko key aur transaction_type_desc ko value rakh rha hai:

chargeback_dict = (
    CurCallidusReport[
        CurCallidusReport['transaction_type_desc']
        .str.contains('charge', case=False, na=False)
    ]
    .set_index('esn')['transaction_type_desc']
    .to_dict()
)

In [58]:
chargeback_dict

{353994916463778: 'Commission Chargeback',
 356484793982097: 'Commission Chargeback',
 358926211157999: 'Commission Chargeback',
 352850710831019: 'Commission Chargeback'}

In [55]:
CurCallidusReport.info()


<class 'pandas.DataFrame'>
RangeIndex: 1174 entries, 0 to 1173
Data columns (total 52 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ï»¿marketid              1174 non-null   str    
 1   custno                   1174 non-null   str    
 2   username                 1093 non-null   str    
 3   name                     1093 non-null   str    
 4   lastname                 1093 non-null   str    
 5   periodname               1174 non-null   str    
 6   dealerpositionname       1174 non-null   str    
 7   dealerfullname           1174 non-null   str    
 8   dealerregionname         0 non-null      float64
 9   dealeraddress            0 non-null      float64
 10  doorpositionname         1174 non-null   int64  
 11  doorfullname             1174 non-null   str    
 12  doorpayeeid              1174 non-null   int64  
 13  doorregionname           0 non-null      float64
 14  dooraddress              0 non-null

In [30]:
#CurCallidusReport file me each unique esn ke hisab se transaction_amount ka sum nikal lia:
transaction_per_esn = CurCallidusReport.groupby('esn', as_index=False)['transaction_amount'].sum()

print(transaction_per_esn)
#jo esn=0 hai vo special revenue hai:

                 esn  transaction_amount
0                  0               62.50
1     16441000364925               16.30
2     16445005172837               22.50
3     16519002875988                0.00
4     16604003005502               67.50
..               ...                 ...
254  864644064152119               81.50
255  866080075751433               68.75
256  866690080277768                0.00
257  866690080411227                0.00
258  999999999999999              690.40

[259 rows x 2 columns]


In [32]:
# ab hum fin dkrege ki useful_cols_ActivationDetailReport['serial'] vs transaction_per_esn['esn'] me kitne common entries hai:

common_count = useful_cols_ActivationDetailReport['serial'].isin(
    transaction_per_esn['esn']
).sum()

print("Common entries:", common_count)

Common entries: 218


In [35]:
# Ab dekhege ki konse serial number jo useful_cols_ActivationDetailReport me hai vo transaction_per_esn me nahi hai:

unmatched = useful_cols_ActivationDetailReport[
    ~useful_cols_ActivationDetailReport['serial'].isin(transaction_per_esn['esn'])
]

print(unmatched[['serial']].head())

Empty DataFrame
Columns: [serial]
Index: []


In [43]:
#ab hum matching serial number ke hisab se transaction_amount ko useful_cols_ActivationDetailReport me naya column Compensation me le aate hai:
    
useful_cols_ActivationDetailReport['Compensation'] = (
    useful_cols_ActivationDetailReport['serial']
    .map(
        transaction_per_esn
        .set_index('esn')['transaction_amount']
    )
    .astype(float)
    .round(2)
)

In [44]:
useful_cols_ActivationDetailReport['Compensation'] = (
    useful_cols_ActivationDetailReport['Compensation']
    .round(2)
)
print(useful_cols_ActivationDetailReport['Compensation'].dtype)

float64


In [45]:
useful_cols_ActivationDetailReport.head()

,storeid,company,serial,accounttype,plancode,plantype1,Compensation
0,C-70812973,DKS WIRELESS 2128007-DCWP,356553100153822,Prime,50UNLG1,Voice,9.30
2,C-70812973,DKS WIRELESS 2128007-DCWP,354445470898494,Prime,BFLX50,Voice,52.00
5,C-70812973,DKS WIRELESS 2128007-DCWP,357917872023779,Prime,NWBYOD30,Voice,30.00
7,C-70812973,DKS WIRELESS 2128007-DCWP,16758008762556,Prime,TAB15SD,Voice,26.25
8,C-70812973,DKS WIRELESS 2128007-DCWP,358926211138619,Prime,BFLX50AAL,Voice,53.00


In [46]:
# check unique values in Compensation column
print(useful_cols_ActivationDetailReport['Compensation'].unique())

[  9.3   52.    30.    26.25  53.    49.5   88.25  27.5   69.5    0.
  89.15  84.75  28.    41.75  96.05  32.75 111.5   92.75  33.6   18.5
  54.75  98.75  27.1   69.    24.35  90.05  36.6   67.1  103.75  46.05
  56.3   23.8   59.    54.5   27.25  71.25  39.75  13.5   60.75  34.5
  38.75  52.32  55.    81.5   31.67  22.5   63.2   32.7   43.6   63.9
  17.    41.5   43.95  75.45  17.7   32.46  13.    74.5   82.3   15.25
   6.5   25.75  16.3   17.75  29.    95.3   10.    80.5   19.5   70.1
  61.5    9.5  100.5   62.8   22.25  71.    43.5   81.75  48.5   84.9
  58.5   67.5   59.95  28.34  16.    46.4   83.6   27.85  57.25  42.5
  65.    35.    85.    18.75  33.    92.    81.85  46.1   98.    87.5
  66.25  78.7   57.1   63.55  59.2 ]


In [48]:
#Now moving towards callidus report file 
CallidusReport = pd.read_csv(r"D:\i\uploads\raw\CallidusDetail_7GT0TS8QU.csv",encoding="latin1")

In [ ]:
CallidusReport.info()
# esn:int64             
# transaction_amount:float64

<class 'pandas.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 45 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   marketid                 153 non-null    str    
 1   custno                   153 non-null    str    
 2   username                 153 non-null    str    
 3   name                     153 non-null    str    
 4   lastname                 153 non-null    str    
 5   periodseq                0 non-null      float64
 6   periodname               153 non-null    str    
 7   dealerpositionname       153 non-null    str    
 8   dealerfullname           153 non-null    str    
 9   dealerpayeeid            153 non-null    str    
 10  dealerparticipantseq     0 non-null      float64
 11  dealerpositionseq        0 non-null      float64
 12  dealerregionname         0 non-null      float64
 13  dealeraddress            0 non-null      float64
 14  doorpositionname         153 non-null

In [52]:
# ab hum matching serial number ke hisab se transaction_amount ko useful_cols_ActivationDetailReport me naya column Compensation me le aate hai:

useful_cols_ActivationDetailReport['Compensation'] = (
    useful_cols_ActivationDetailReport['serial']
    .map(
        CallidusReport.set_index('esn')['transaction_amount']
    )
    .fillna(0.00)
    .astype(float)
    .round(2)
)

In [53]:
useful_cols_ActivationDetailReport.head()

,storeid,company,serial,accounttype,plancode,plantype1,Compensation
0,C-70812973,DKS WIRELESS 2128007-DCWP,356553100153822,Prime,50UNLG1,Voice,0.0
2,C-70812973,DKS WIRELESS 2128007-DCWP,354445470898494,Prime,BFLX50,Voice,204.0
5,C-70812973,DKS WIRELESS 2128007-DCWP,357917872023779,Prime,NWBYOD30,Voice,0.0
7,C-70812973,DKS WIRELESS 2128007-DCWP,16758008762556,Prime,TAB15SD,Voice,118.2
8,C-70812973,DKS WIRELESS 2128007-DCWP,358926211138619,Prime,BFLX50AAL,Voice,238.0


In [59]:
# chargeback_dict se map karke pata chalega ki kaunse serial number chargeback hai:

useful_cols_ActivationDetailReport['Chargeback'] = (
    useful_cols_ActivationDetailReport['serial']
    .map(chargeback_dict)
)

In [60]:
useful_cols_ActivationDetailReport.head()

,storeid,company,serial,accounttype,plancode,plantype1,Compensation,Chargeback
0,C-70812973,DKS WIRELESS 2128007-DCWP,356553100153822,Prime,50UNLG1,Voice,0.0,NaN
2,C-70812973,DKS WIRELESS 2128007-DCWP,354445470898494,Prime,BFLX50,Voice,204.0,NaN
5,C-70812973,DKS WIRELESS 2128007-DCWP,357917872023779,Prime,NWBYOD30,Voice,0.0,NaN
7,C-70812973,DKS WIRELESS 2128007-DCWP,16758008762556,Prime,TAB15SD,Voice,118.2,NaN
8,C-70812973,DKS WIRELESS 2128007-DCWP,358926211138619,Prime,BFLX50AAL,Voice,238.0,NaN
